In [8]:
from openai import OpenAI

openai_client= OpenAI()



In [9]:
from gitsource import GithubRepositoryDataReader, chunk_documents
reader= GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md","mdx"}
)
files=reader.read()

parsed_docs=[doc.parse() for doc in files]
chunked_docs= chunk_documents(parsed_docs,size=3000,step= 1500)



In [10]:
from minsearch import AppendableIndex

index=AppendableIndex(
    text_fields=["title","description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)



In [11]:
def search(query):
    results= index.search(
        query=query,
        num_results=5
    )
    return results



In [12]:
#setup rag prompt template
import json
RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context=json.dumps(search_results,indent=2),
    return RAG_PROMPT_TEMPLATE.format(question=question,context=context)




In [13]:
#make a query and build the prompt
question="how do i create a dashbord in evidently"
search_results=search(question)
user_prompt=build_prompt(question, search_results)

#call the LLM
messages=[
    {"role":"system", "content":RAG_INSTRUCTIONS},
    {"role":"user", "content":user_prompt}
]

response=openai_client.responses.create(
    model='gpt-5.4-nano',
    input=messages
)
print (response.output_text)

The provided documentation context doesn’t explain how to create a **dashboard** in Evidently (it only mentions that the Evidently Platform offers “rich dashboards,” but gives no step-by-step instructions).

If you tell me whether you’re using **Evidently Cloud UI** or **self-hosted Evidently Platform**, I can look for the relevant setup steps—but with the current context, I can’t provide accurate dashboard-creation instructions.


In [14]:
# lets make it agentic
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-5.4-nano',
    input=messages,
)

response.usage.input_tokens



59

In [15]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response=openai_client.responses.create(
    model='gpt-5.4-nano',
    input=messages,
    tools=[search_tool]
)

response.usage.input_tokens



112

In [16]:
#handling the tool call
tool_call=response.output[0]
tool_call

ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_eoWdqP31lTfCiyVwTYSWOHlQ', name='search', type='function_call', id='fc_0ed2f9419587d25c006ad3822701d887d299511ceb297b3416', async_=None, caller=None, namespace=None, status='completed')

In [17]:
messages.append(tool_call)
print(tool_call.arguments)
arguments=json.loads(tool_call.arguments)
print(arguments)
#execute the searh with those arguments
#explictly first

search_results=search(**arguments)
search_results[:1]

{"query":"create dashboard in Evidently"}
{'query': 'create dashboard in Evidently'}


[{'start': 0,
  'content': "<Check>\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n## What is a Dashboard?\n\nA Dashboard provides a clear view of your AI application performance. You can use it:\n\n- to track evaluation results across multiple experiments;\n- to track live production quality over time.\n\nEach Project has its own Dashboard. It's empty at first.\n\nTo populate it, you need to run an evaluation and **save at least one Report** to the Project. You can then choose values from Reports to plot.\n\n![](/images/dashboard_llm_light.png)\n\n## Dashboard Tabs\n\n<Check>\n  Multiple Tabs are available in **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\nYou can logically organize Panels within the same Dashboard into different Tabs.\n\n![](/images/dashboard_llm_tabs.gif)\n\n## **Dashboard Panels**\n\nA Panel is a visual element in the Dashboard that displays specified values in a single widget. Panels ca

In [18]:
#now we need to call the tool as indicated by AI
call_output= {
    "type":"function_call_output",
    "call_id":tool_call.call_id,
    "output": json.dumps(search_results)
}
print (call_output)
messages.append(call_output)

#now finall call


{'type': 'function_call_output', 'call_id': 'call_eoWdqP31lTfCiyVwTYSWOHlQ', 'output': '[{"start": 0, "content": "<Check>\\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\\n</Check>\\n\\n## What is a Dashboard?\\n\\nA Dashboard provides a clear view of your AI application performance. You can use it:\\n\\n- to track evaluation results across multiple experiments;\\n- to track live production quality over time.\\n\\nEach Project has its own Dashboard. It\'s empty at first.\\n\\nTo populate it, you need to run an evaluation and **save at least one Report** to the Project. You can then choose values from Reports to plot.\\n\\n![](/images/dashboard_llm_light.png)\\n\\n## Dashboard Tabs\\n\\n<Check>\\n  Multiple Tabs are available in **Evidently Cloud** and **Evidently Enterprise**.\\n</Check>\\n\\nYou can logically organize Panels within the same Dashboard into different Tabs.\\n\\n![](/images/dashboard_llm_tabs.gif)\\n\\n## **Dashboard Pan

In [19]:
response = openai_client.responses.create(
    model='gpt-5.4-nano',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

3693

In [20]:
#now lets add structured output
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [21]:
response = openai_client.responses.parse(
    model='gpt-5.4-nano',
    input=messages,
    tools=[search_tool],
    text_format=RAGResponse
)

response.usage.input_tokens

3916

In [22]:
rag_response=response.output_parsed
print(rag_response)

answer='You can create an **Evidently Dashboard** per Project, then populate it with **Panels** based on metrics from **Reports**.\n\n## 1) Make sure the Dashboard has data (Reports)\n- Each Project has its own Dashboard (it’s empty at first).\n- To populate it, you must **run an evaluation and save at least one Report to the Project**.\n- Then you can create Panels by selecting metric values that come from the saved Reports.\n\n## 2) Add Panels (Dashboard “as code” via Python API)\nDashboards as code are available in **Evidently OSS, Cloud, and Enterprise**.\n\n### Add a tab (optional)\n```python\nproject.dashboard.add_tab("Another Tab")\n```\n\n### Add a Panel\nYou add Panels to a specific tab (the tab can be created if it doesn’t exist).\n\n**Example: Text panel**\n```python\nfrom evidently.sdk.models import PanelMetric\nfrom evidently.sdk.panels import DashboardPanelPlot\n\nproject.dashboard.add_panel(\n    DashboardPanelPlot(\n        title="Dashboard title",\n        size="full",

In [23]:
#define method to call tools

def make_call(tool_call):
    arguments=json.loads(tool_call.arguments)
    name=tool_call.name

    if name=='search':
        result = search(**arguments)
    elif name =='add_entry':
        result = add_entry(**arguments)
    else:
        result='not found "{name}"'    
    return {
        "type":"function_call_output",
        "call_id":tool_call.call_id,
        "output" : json.dumps(result)
    }



In [24]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

IMPORTANT: When you explore the knowledge base, make at least 3 different
searchers to make sure you explore the topic well.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [25]:
question = "How do I create a dahsbord in Evidently?"

In [26]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [27]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=message_history,
    tools=[search_tool],
)

In [28]:
message_history.extend(response.output)

In [29]:
message_history

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nIMPORTANT: When you explore the knowledge base, make at least 3 different\nsearchers to make sure you explore the topic well.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to \ninclude the word 'evidently' in search results\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"dashboard create dashboard Evidently"}', call_id='call_7I3MhBbypWmUaS75c3q1Aq9L', name='search', type='function_call', id='fc_0b8ad38df0b9d78d006ad38231655c87d2b1c7092e316871d7', async_=None, caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"workspace dashboard report UI create"}', call_id='call_UqGcra0Gidjk6lRGqUl2DTSV', name='se

In [30]:
for message in response.output:
    if message.type=='function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output=make_call(message)
        message_history.append(tool_call_output)

executing search({"query":"dashboard create dashboard Evidently"})...
executing search({"query":"workspace dashboard report UI create"})...
executing search({"query":"build dashboard from report display share"})...


In [31]:
len(message_history)

8

In [32]:
#make another request to get the final answer
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=message_history,
    tools=[search_tool],
)
response.usage.input_tokens

11702

In [33]:
print(response.output_text)

To create a dashboard in Evidently, you need a **Project** with at least one saved **Report** first. The Dashboard uses values from Reports as its data source.

## Basic steps

1. **Connect to Evidently Cloud**
2. **Create a Project**
3. **Run an evaluation and save at least one Report to the Project**
4. **Add Panels** to the Dashboard
5. Optionally **organize Panels into Tabs**

## Two ways to build it

### 1) In the UI
- Open the Dashboard
- Enter **Edit** mode
- Click **Add Panel**
- Configure the panel
- Click **Save**
- Choose the Tab where you want it

You can also:
- Click **add Tab** to create a new Tab
- Use **empty** to create a custom Tab name

### 2) Using Python API
Import the needed classes and add Panels to a Project dashboard:

```python
from evidently.sdk.models import PanelMetric
from evidently.sdk.panels import DashboardPanelPlot
```

Example of adding a text panel:

```python
project.dashboard.add_panel(
    DashboardPanelPlot(
        title="Dashboard title",
    

In [34]:
#agentic loop
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [35]:
#define agentic loop
message_history=[
    {"role":"system", "content":instructions},
    {"role":"user", "content":question}
]

iteration_number=1

while True:
        response = openai_client.responses.create(
        model='gpt-5.4-nano',
        input=message_history,
        tools=[search_tool],
    )
        print(f'iteration mnumber  {iteration_number}')

        message_history.extend(response.output)

        has_function_calls=False

        for message in response.output:
                if message.type== 'function_call':
                        print (f'executing {message.name} ({message.arguments})')
                        tool_call_output=make_call(message)
                        message_history.append(tool_call_output)
                        has_function_calls=True
                               
                if message.type=='message':
                        text=message.content[0].text
                        print ('ASSISTANT:', text)
        iteration_number= iteration_number + 1
        print()

        if not has_function_calls:
                break

iteration mnumber  1
ASSISTANT: Iteration 1 — Search
I’m going to search the knowledge base for the specific instructions on creating a dashboard, because the user asks “How do I create a dashboard,” and we should find the canonical setup/steps in the docs. This query should return the most direct “dashboard creation” guide or API references.
executing search ({"query":"create dashboard in Evidently"})

iteration mnumber  2
ASSISTANT: Iteration 2 — Analyze + more searches
From the first search, we have the key concept: dashboards exist per Project, start empty, and get populated by saving at least one Report; panels can be added via Python API or (in Cloud/Enterprise) via UI. Next I want to find the exact steps for “add dashboard panels” via the API and also how the UI differs (Cloud/Enterprise), so the user can follow the right approach.
executing search ({"query":"dashboard add panels UI Cloud Enterprise add panels directly from interface"})
executing search ({"query":"project.dashbo

In [36]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}


In [37]:
#multiple tools working together
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]
tools=[search_tool,add_entry_tool]
iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 1...
ASSISTANT: To answer your question about creating a dashboard, I'll start by searching for any existing documentation related to dashboard creation. This will help me understand the steps or guidelines provided for creating dashboards and ensure that I give you accurate information.

Let me perform that search now.
executing search({"query":"create dashboard"})...

iteraration number 2...
ASSISTANT: Based on the initial search results, I found some useful information on creating dashboards and adding panels. I will now analyze this information to summarize the main steps involved and check if there's more detail I can gather, especially about the specific methods available for creating dashboards through both the Python API and the user interface.

I'll perform two additional searches: one focusing on adding tabs and a second one looking specifically at dashboard configuration options. This should give a more comprehensive view of the dashboard creation process.

In [38]:
message_history.append(
    {"role": "user", "content": "add this content to our database"}
)

In [39]:
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 4...
executing add_entry({"filename":"dashboard_creation_guide.mdx","title":"Creating a Dashboard","description":"Step-by-step guide on how to create a dashboard in Evidently.","content":"### Step 1: Understand Dashboards\nA dashboard allows you to visualize evaluation results over time, making it crucial to have Reports with evaluation data in your project.\n\n### Step 2: Adding Tabs\nTabs help organize panels in your dashboard. You can add multiple tabs for logical categorization:\n- **To add a new tab**:\n  - Enter \"Edit\" mode on the Dashboard.\n  - Click the plus sign to add a Tab.\n  - Choose to create a custom tab or use pre-built templates.\n\n### Step 3: Adding Panels\nPanels display metrics and can be of various types:\n1. **Enter \"Edit\" mode**.\n2. Click \"Add Panel\".\n3. Choose the type of panel you want (text, counter, pie chart, line plot, etc.).\n4. Define the metrics and other parameters (like size and tags).\n5. Save the panel to the desired tab.

In [40]:
index.docs[-1]

{'start': 0,
 'content': '### Step 1: Understand Dashboards\nA dashboard allows you to visualize evaluation results over time, making it crucial to have Reports with evaluation data in your project.\n\n### Step 2: Adding Tabs\nTabs help organize panels in your dashboard. You can add multiple tabs for logical categorization:\n- **To add a new tab**:\n  - Enter "Edit" mode on the Dashboard.\n  - Click the plus sign to add a Tab.\n  - Choose to create a custom tab or use pre-built templates.\n\n### Step 3: Adding Panels\nPanels display metrics and can be of various types:\n1. **Enter "Edit" mode**.\n2. Click "Add Panel".\n3. Choose the type of panel you want (text, counter, pie chart, line plot, etc.).\n4. Define the metrics and other parameters (like size and tags).\n5. Save the panel to the desired tab.\n\n**Example code (using Python API) to add a panel:**\n```python\nproject.dashboard.add_panel(\n    DashboardPanelPlot(\n        title="Row count",\n        subtitle="Total number of ev

In [41]:
#the q&a loop
tools=[search_tool,add_entry_tool]

message_history=[
    {"role":"system", "content":instructions}
]
iteration_number=1
while True:
    user_prompt=input('You:')
    if user_prompt.lower()=='stop':
        break
    message_history.append({"role":"user", "content":user_prompt})
    while True:
        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=message_history,
            tools=tools,
        )
        print(f'iteraration number {iteration_number}...') 
        message_history.extend(response.output)

        has_function_calls = False
        for message in response.output:
            if message.type == 'function_call':
                print(f'executing {message.name}({message.arguments})...')
                tool_call_output = make_call(message)
                message_history.append(tool_call_output)
                has_function_calls = True

            if message.type == 'message':
                text = message.content[0].text
                print('ASSISTANT:', text)

        iteration_number = iteration_number + 1
        print()

        if not has_function_calls:
            break

 

iteraration number 1...
ASSISTANT: It seems your message was empty. Please provide a question or topic you'd like information on, and I'll assist you accordingly!



In [42]:
class Agent:

    def __init__(self, llm_client, model, instructions, tools):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name

        if name == 'search':
            result = search(**arguments)
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else:
            result = 'not found tool "{name}"'

        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }   

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]

        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0

        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )

            print(f'iteraration number {iteration_number}...') 
            message_history.extend(response.output)

            has_function_calls = False

            for message in response.output:
                if message.type == 'function_call':
                    print(f'executing {message.name}({message.arguments})...')
                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)
                    has_function_calls = True

                if message.type == 'message':
                    text = message.content[0].text
                    print('ASSISTANT:', text)

            iteration_number = iteration_number + 1
            print()

            if not has_function_calls:
                break

        return message_history        

    def qna(self):
        message_history = [
            {"role": "system", "content": instructions},
        ]

        iteration_number = 1

        # Q&A loop
        while True:
            user_prompt = input('You:')
            if user_prompt.lower().strip() == 'stop':
                break

            message_history = self.loop(user_prompt, message_history)

In [43]:
agent = Agent(
    llm_client=OpenAI(),
    model='gpt-5.4-nano',
    instructions=instructions,
    tools=tools
)
messages = agent.loop('evidently adshboards')


iteraration number 0...
executing search({"query":"adshboards"})...

iteraration number 1...
ASSISTANT: I searched for the term “adshboards” because it likely relates to dashboards in the Evidently documentation and could be a misspelling of the actual keyword. Since the initial search returned nothing, I’ll broaden the query to likely correct terms next.


executing search({"query":"dashboards"})...
executing search({"query":"ads dashboards"})...

iteraration number 2...
ASSISTANT: It looks like you meant **dashboard(s)** (your query “evidently adshboards” didn’t match anything in the docs). Here’s what the Evidently documentation says about building dashboards.

## Add Tabs to a Dashboard
You can organize panels into multiple tabs:
- **UI**: open the Dashboard, go to **Edit** mode, then use **add Tab**. To make a custom tab, choose the **empty** tab and name it.
- **Python**: use `project.dashboard.create_tab("My tab")` and then `project.save()`.

## Add Panels to a Dashboard
Panels 

In [44]:
#structured output
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [45]:
class StructuredAgent1(Agent):

    def __init__(self, llm_client, model, instructions, tools, output_type):
        super().__init__(llm_client, model, instructions, tools)
        self.output_type = output_type

    def structured_loop(self, user_prompt, message_history=None):
        message_history = self.loop(user_prompt, message_history)
        response = self.llm_client.responses.parse(
            model=self.model,
            input=message_history,
            text_format=self.output_type
        )

        output = response.output_parsed
        return message_history, output

In [46]:
structure_agent_1 = StructuredAgent1(
    llm_client=OpenAI(),
    model='gpt-5.4-nano',
    instructions=instructions,
    tools=[search_tool],
    output_type=RAGResponse
)

_, output = structure_agent_1.structured_loop('bashboards')

iteraration number 0...
executing search({"query":"bashboard"})...

iteraration number 1...
executing search({"query":"bashboard dashboards"})...

iteraration number 2...
executing search({"query":"dashboard create_tab add_panel project.dashboard"})...

iteraration number 3...
ASSISTANT: I couldn’t find anything specifically called “bashboards” in the documentation, so I searched for the closest likely term: **“bashboard dashboards”**. This led me to the dashboard documentation, which explains how to build and configure monitoring dashboards (tabs + panels).

## What you can do in Evidently dashboards

### Add Tabs
- **UI**: Enter **Edit** mode on the Dashboard, click **add Tab**, choose **empty**, and name it.
- **Python**: Create an empty tab with:
```python
project.dashboard.create_tab("My tab")
project.save()
```
- **Alternative (API)**: When adding a panel, you can specify a `tab="..."`; if the tab doesn’t exist, it will be created.

### Add Panels
You can add panels via **UI** or

In [47]:
print(output.answer)

I couldn’t find anything specifically called **“bashboards”** in the documentation.

What I *could* find is documentation for **Dashboards** (tabs + panels). Here’s how to create and populate them:

## Add Tabs
- **UI**: Enter **Edit** mode on the Dashboard → click **add Tab** → choose **empty** → name it.
- **Python**:
```python
project.dashboard.create_tab("My tab")
project.save()
```
- (Also possible) When adding a panel, you can set `tab="..."`; if that tab doesn’t exist, it can be created.

## Add Panels
### UI
- Enter **Edit** mode → **Add panel** → configure → **preview** → **publish/save**.
- To delete/edit a panel: hover it in **Edit** mode and choose the action.

### Python API (dashboards as code)
- You must first connect to a workspace and **create a Project**.
- Then add panels to the dashboard using `project.dashboard.add_panel(...)` and save with `project.save()`.

## Important note
Dashboards pull values from **Reports** that are already added to the Project. If the nee